# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asiya-Akhtar/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue turns the validated signals into practical content recommendations. Actions are ranked so that the strongest observed opportunities are reviewed first. Each recommendation has a reason code so a human can understand why the item was selected.

The main reason codes are:

- STALE: the content has not been refreshed recently.
- LOW_CTR_VISIBLE: the page receives meaningful impressions and has a visible search position, but its observed CTR is low.
- STALE_AND_LOW_CTR: both signals are present, making the item a stronger review candidate.

These recommendations are decision-support, not automatic publishing instructions.

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/Asiya-Akhtar/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [15]:
# Current-state signals only
df["stale"] = df["days_since_last_update"] >= 90

df["low_ctr_visible"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] >= 1)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

df["action_score"] = (
    2 * df["stale"].astype(int)
    + df["low_ctr_visible"].astype(int)
)

df["reason_code"] = np.select(
    [
        df["stale"] & df["low_ctr_visible"],
        df["stale"],
        df["low_ctr_visible"],
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE",
        "LOW_CTR_VISIBLE",
    ],
    default="NO_FLAG"
)

df["action"] = np.select(
    [
        df["stale"] & df["low_ctr_visible"],
        df["stale"],
        df["low_ctr_visible"],
    ],
    [
        "Review and refresh content",
        "Review for freshness update",
        "Review title/snippet for CTR improvement",
    ],
    default="Monitor"
)

queue = df.sort_values(
    ["action_score", "impressions_90d"],
    ascending=[False, False]
).copy()

print(queue[
    ["action_score", "reason_code", "action"]
].head(10))

       action_score        reason_code                      action
6653              3  STALE_AND_LOW_CTR  Review and refresh content
26531             3  STALE_AND_LOW_CTR  Review and refresh content
3394              3  STALE_AND_LOW_CTR  Review and refresh content
26255             3  STALE_AND_LOW_CTR  Review and refresh content
7445              3  STALE_AND_LOW_CTR  Review and refresh content
19173             3  STALE_AND_LOW_CTR  Review and refresh content
26474             3  STALE_AND_LOW_CTR  Review and refresh content
9200              3  STALE_AND_LOW_CTR  Review and refresh content
7133              3  STALE_AND_LOW_CTR  Review and refresh content
19304             3  STALE_AND_LOW_CTR  Review and refresh content


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for content teams and analysts who need to prioritize pages for manual review. It is a decision-support tool that helps identify pages showing observed freshness or CTR-related signals.

The recommendations are intended to help decide which pages should be reviewed first. They do not prove that a page needs a specific change, and they do not establish that one signal caused a performance outcome.

The playbook is not intended for automatic publishing, automatic deletion of content, or unsupervised changes to important pages. Recommendations should be reviewed against the actual page, search intent, business context, and current content before action is taken.

The results can become stale as content, rankings, search behavior, and traffic change.

In [16]:
print("Action counts:")
print(queue["action"].value_counts())

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())


Action counts:
action
Monitor                                     14439
Review title/snippet for CTR improvement     6216
Review for freshness update                  5816
Review and refresh content                   3529
Name: count, dtype: int64

Reason-code counts:
reason_code
NO_FLAG              14439
LOW_CTR_VISIBLE       6216
STALE                 5816
STALE_AND_LOW_CTR     3529
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before taking action, a human should review:

1. The actual page and its current content.
2. Whether the page still matches the search intent.
3. Whether the observed ranking and CTR signals are current.
4. Whether the page is strategically important.
5. Whether a refresh would improve the content rather than simply changing it.
6. Whether the recommendation conflicts with current business or editorial requirements.

### No-go cases

The system should not automatically:

- publish content;
- delete or redirect pages;
- rewrite important pages without review;
- change claims, facts, prices, or legal information;
- make decisions based only on the score;
- treat correlation as proof of causation.

The queue is for prioritization and decision-support, not autonomous content management.

In [17]:
review_required = queue[
    queue["reason_code"] != "NO_FLAG"
].copy()

print("Items requiring human review:", len(review_required))
print("\nTop 10 review candidates:")
print(
    review_required[
        ["action_score", "reason_code", "action",
         "days_since_last_update", "impressions_90d",
         "avg_position", "ctr"]
    ].head(10)
)

Items requiring human review: 15561

Top 10 review candidates:
       action_score        reason_code                      action  \
6653              3  STALE_AND_LOW_CTR  Review and refresh content   
26531             3  STALE_AND_LOW_CTR  Review and refresh content   
3394              3  STALE_AND_LOW_CTR  Review and refresh content   
26255             3  STALE_AND_LOW_CTR  Review and refresh content   
7445              3  STALE_AND_LOW_CTR  Review and refresh content   
19173             3  STALE_AND_LOW_CTR  Review and refresh content   
26474             3  STALE_AND_LOW_CTR  Review and refresh content   
9200              3  STALE_AND_LOW_CTR  Review and refresh content   
7133              3  STALE_AND_LOW_CTR  Review and refresh content   
19304             3  STALE_AND_LOW_CTR  Review and refresh content   

       days_since_last_update  impressions_90d  avg_position   ctr  
6653                      104           517715           4.2  0.14  
26531                     10

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored because the underlying signals can change over time.

A review should be triggered if:

- the distribution of freshness values changes substantially;
- impressions or CTR distributions shift;
- the proportion of flagged pages changes unexpectedly;
- observed performance of recommended actions becomes weaker;
- the data schema or definitions of the signals change;
- new data becomes available that changes the modeling assumptions.

A model or rule should be reconsidered rather than automatically retrained simply because time has passed. Retraining or rule revision should be based on observed evidence that the current recommendations are becoming less useful.

Monitoring is intended to identify drift and stale assumptions, not to guarantee future performance.

In [19]:
monitoring_summary = pd.DataFrame({
    "metric": [
        "rows",
        "stale_rate",
        "low_ctr_visible_rate",
        "flagged_rate"
    ],
    "value": [
        len(df),
        df["stale"].mean(),
        df["low_ctr_visible"].mean(),
        (df["reason_code"] != "NO_FLAG").mean()
    ]
})

print(monitoring_summary)

                 metric         value
0                  rows  30000.000000
1            stale_rate      0.311500
2  low_ctr_visible_rate      0.324833
3          flagged_rate      0.518700


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported so that the recommendations section of the research paper can trace back to the notebook output.

The exported queue contains the ranking score, reason code, action label, and supporting current-state signals needed for human review.

The CSV is regenerated when the notebook runs and is therefore treated as a derived output rather than a source dataset.

In [20]:
output_dir = Path("../../work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "action_playbook_queue.csv"

export_columns = [
    "action_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue[export_columns].to_csv(output_path, index=False)

print("Exported:", output_path)
print("Rows exported:", len(queue))

Exported: ../../work/outputs/action_playbook_queue.csv
Rows exported: 30000


In [21]:
check = pd.read_csv(output_path)

print("Export exists:", output_path.exists())
print("Export rows:", len(check))
print("Export columns:", check.columns.tolist())

display(check.head(10))

Export exists: True
Export rows: 30000
Export columns: ['action_score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']


,action_score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
0,3,STALE_AND_LOW_CTR,Review and refresh content,104,517715,4.2,0.14
1,3,STALE_AND_LOW_CTR,Review and refresh content,104,309910,5.6,0.16
2,3,STALE_AND_LOW_CTR,Review and refresh content,104,295097,7.3,0.05
3,3,STALE_AND_LOW_CTR,Review and refresh content,104,211366,5.1,0.41
4,3,STALE_AND_LOW_CTR,Review and refresh content,104,208678,9.7,0.00
5,3,STALE_AND_LOW_CTR,Review and refresh content,104,201584,5.8,0.24
6,3,STALE_AND_LOW_CTR,Review and refresh content,104,201111,5.7,0.11
7,3,STALE_AND_LOW_CTR,Review and refresh content,104,192205,12.5,0.24
8,3,STALE_AND_LOW_CTR,Review and refresh content,104,190623,4.3,0.24
9,3,STALE_AND_LOW_CTR,Review and refresh content,104,187893,4.0,0.45


### Self-check

- [x] Ranked actions are produced.
- [x] Every recommendation has a reason code.
- [x] Intended use and limits are stated.
- [x] Human review requirements are stated.
- [x] No-go cases are documented.
- [x] Monitoring and retrain triggers are described.
- [x] The ranked queue is exported to `work/outputs/`.
- [x] The notebook uses careful decision-support language.
- [ ] Notebook has been run top to bottom successfully.
- [ ] Notebook has been committed to the repository.